# 06 · LLM Router：TEU 工具、规则预审节点与澄清

用户输入先由 `llm_router` 做结构化意图分析，再选择一条业务分支：

`User input → llm_router → (TEU tool | rule precheck node | clarify) → response_llm → END`

- `calculate_teu`：沿用 05 的 `@tool` TEU 计算能力。
- `rule_precheck_node`：模拟调用规则预审服务，返回可用于课堂演示的 mock 结果。
- `clarify_node`：当箱型、数量或订舱 ID 不足时，为后续 LLM 准备澄清问题。

三条分支都不直接面向用户输出；`response_llm` 根据分支结果统一组织最终回答。


In [ ]:
from __future__ import annotations

import os
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv


def find_repo_root() -> Path:
    """向上找到训练仓根目录，避免 Notebook 工作目录变化。"""
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
            return candidate
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

missing_llm = [
    name for name in ("LLM_BASE_URL", "LLM_MODEL")
    if not (os.getenv(name) or "").strip()
]
if missing_llm:
    raise ValueError("真实 LLM Router 必须配置 .env，缺少：" + ", ".join(missing_llm))

print("cwd =", ROOT)
print("mode = live LLM router required")


In [ ]:
def show_graph(graph):
    """在 Notebook 中展示 State / Node / Edge。"""
    from IPython.display import Image, display

    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception as exc:
        print("PNG 不可用，打印 Mermaid：", exc)
        print(graph.get_graph().draw_mermaid())


In [ ]:
import json
import operator
from typing import Annotated, Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field


RouterIntent = Literal["calculate_teu", "rule_precheck", "clarify"]
RouteKey = Literal["calculate_teu", "rule_precheck", "clarify"]


class RouteDecision(BaseModel):
    """LLM Router 只做意图分类和参数提取。"""

    intent: RouterIntent = Field(description="TEU 计算、规则预审或需要澄清")
    equipment: str | None = Field(default=None, description="TEU 计算的箱型")
    quantity: int | None = Field(default=None, description="TEU 计算的箱数")
    booking_id: str | None = Field(default=None, description="规则预审的订舱 ID")
    clarification_question: str | None = Field(
        default=None,
        description="信息不足时，需要向用户询问的一个问题",
    )


class RouteState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    router_output: AIMessage
    route: RouteDecision
    selected_route: RouteKey
    business_result: dict
    status: str
    trace: Annotated[list[str], operator.add]


TEU_PER_EQUIPMENT = {"20GP": 1, "40GP": 2, "40HQ": 2}
SERVICE_CALLS = defaultdict(int)


@tool
def calculate_teu(equipment: str, quantity: int) -> int:
    """按课堂约定换算 TEU。支持 20GP、40GP、40HQ。"""
    key = equipment.strip().upper()
    if key not in TEU_PER_EQUIPMENT:
        raise ValueError(f"不支持的箱型: {equipment}")
    if quantity < 0:
        raise ValueError("quantity 不能为负数")
    return TEU_PER_EQUIPMENT[key] * quantity


@tool("rule_precheck")
def select_rule_precheck(booking_id: str) -> str:
    """当用户要求对订舱做规则预审时，选择规则预审节点。"""
    # 该工具只向 Router 提供意图 schema，图中不会直接执行它。
    return booking_id


def mock_rule_precheck_service(booking_id: str) -> dict:
    """模拟外部规则预审服务，课堂环境不发起真实网络请求。"""
    SERVICE_CALLS[booking_id] += 1
    return {
        "mock": True,
        "booking_id": booking_id,
        "decision": "PASS",
        "risk_level": "LOW",
        "matched_rules": ["MOCK-RULE-001"],
        "remarks": "课堂 mock：未发现阻断项",
    }


def make_model():
    """通过统一入口创建 OpenAI-compatible 模型，不直接导入 ChatOpenAI。"""
    return init_chat_model(
        model=os.environ["LLM_MODEL"],
        model_provider="openai",
        base_url=os.environ["LLM_BASE_URL"],
        api_key=os.getenv("LLM_API_KEY") or "not-required",
        temperature=0,
    )


base_model = make_model()
router_model = base_model.bind_tools([calculate_teu, select_rule_precheck])


def llm_router(state: RouteState) -> dict:
    response = router_model.invoke([
        SystemMessage(content=(
            "你是业务意图路由器，只做分类和参数提取，不回答用户。"
            "TEU 或箱量换算调用 calculate_teu，必须同时提取 equipment 和 quantity；"
            "订舱规则预审调用 rule_precheck，必须提取 booking_id；"
            "意图不清或必需参数不完整时不调用工具，只输出一个简洁的澄清问题。"
        )),
        *state["messages"],
    ])

    if response.tool_calls:
        tool_call = response.tool_calls[0]
        args = tool_call["args"]
        if tool_call["name"] == "calculate_teu":
            decision = RouteDecision(
                intent="calculate_teu",
                equipment=args.get("equipment"),
                quantity=args.get("quantity"),
            )
        elif tool_call["name"] == "rule_precheck":
            decision = RouteDecision(
                intent="rule_precheck",
                booking_id=args.get("booking_id"),
            )
        else:
            decision = RouteDecision(intent="clarify")
    else:
        router_text = response.content if isinstance(response.content, str) else str(response.content)
        decision = RouteDecision(
            intent="clarify",
            clarification_question=router_text.strip() or None,
        )

    return {
        "router_output": response,
        "route": decision,
        "selected_route": decision.intent,
        "trace": [f"llm_router:{decision.intent}"],
    }


def teu_tool_node(state: RouteState) -> dict:
    decision = state["route"]
    if decision.equipment is None or decision.quantity is None:
        raise ValueError("TEU 路由缺少 equipment 或 quantity")
    equipment = decision.equipment.strip().upper()
    teu = calculate_teu.invoke({"equipment": equipment, "quantity": decision.quantity})
    return {
        "status": "success",
        "business_result": {
            "type": "teu_calculation",
            "equipment": equipment,
            "quantity": decision.quantity,
            "teu": teu,
        },
        "trace": ["teu_tool:calculate_teu"],
    }


def rule_precheck_node(state: RouteState) -> dict:
    booking_id = state["route"].booking_id
    if not booking_id:
        raise ValueError("规则预审路由缺少 booking_id")
    result = mock_rule_precheck_service(booking_id)
    return {
        "status": "success",
        "business_result": {"type": "rule_precheck", **result},
        "trace": ["rule_precheck_node:mock_service"],
    }


def clarify_node(state: RouteState) -> dict:
    decision = state["route"]
    question = decision.clarification_question
    if not question:
        if decision.intent == "calculate_teu":
            question = "请补充箱型（20GP、40GP 或 40HQ）和箱数。"
        elif decision.intent == "rule_precheck":
            question = "请提供需要预审的订舱 ID。"
        else:
            question = "请问您要计算 TEU，还是进行订舱规则预审？"
    return {
        "selected_route": "clarify",
        "status": "need_clarification",
        "business_result": {"type": "clarification", "question": question},
        "trace": ["clarify_node:need_clarification"],
    }


def response_llm(state: RouteState) -> dict:
    context = json.dumps(state["business_result"], ensure_ascii=False)
    response = base_model.invoke([
        SystemMessage(content=(
            "你是订舱业务助手。根据后端节点结果生成简洁、准确的最终回答，"
            "不得编造结果中没有的信息。如果 type=clarification，只输出一个澄清问题。"
            f"\n后端节点结果：{context}"
        )),
        *state["messages"],
    ])
    return {"messages": [response], "trace": ["response_llm:final"]}


# path_map：路由函数返回值 → 实际节点名
ROUTES: dict[RouteKey, str] = {
    "calculate_teu": "teu_tool",
    "rule_precheck": "rule_precheck",
    "clarify": "clarify",
}


def route_after_llm(state: RouteState) -> RouteKey:
    """只读取 LLM 的结构化决策；同时对必需参数做最后一道防守校验。"""
    decision = state["route"]
    if decision.intent == "calculate_teu":
        if decision.equipment is not None and decision.quantity is not None:
            return "calculate_teu"
        return "clarify"
    if decision.intent == "rule_precheck":
        return "rule_precheck" if decision.booking_id else "clarify"
    return "clarify"


builder = StateGraph(RouteState)
builder.add_node("llm_router", llm_router)
builder.add_node("teu_tool", teu_tool_node)
builder.add_node("rule_precheck", rule_precheck_node)
builder.add_node("clarify", clarify_node)
builder.add_node("response_llm", response_llm)
builder.add_edge(START, "llm_router")
builder.add_conditional_edges("llm_router", route_after_llm, ROUTES)
for node_name in ["teu_tool", "rule_precheck", "clarify"]:
    builder.add_edge(node_name, "response_llm")
builder.add_edge("response_llm", END)

graph = builder.compile()
show_graph(graph)


In [ ]:
cases = [
    ("请计算 2×40HQ 的 TEU", "calculate_teu", "success"),
    ("请预审订舱 BK-DEMO-006 的适用规则", "rule_precheck", "success"),
    ("请帮我计算 TEU", "clarify", "need_clarification"),
]
results_by_route = {}

for question, expected_route, expected_status in cases:
    result = graph.invoke({"messages": [HumanMessage(content=question)], "trace": []})
    final_answer = result["messages"][-1]
    results_by_route[expected_route] = result

    print("Q:", question)
    print("Router protocol:", result["router_output"].tool_calls or result["router_output"].content)
    print("Router decision:", result["route"].model_dump())
    print("Selected route:", result["selected_route"])
    print("Business result:", result["business_result"])
    print("Trace:", " -> ".join(result["trace"]))
    print("Final answer:", final_answer.content)
    print("---")

    assert result["selected_route"] == expected_route
    assert result["status"] == expected_status
    assert isinstance(final_answer, AIMessage)
    assert str(final_answer.content).strip()
    assert result["trace"][-1] == "response_llm:final"

assert results_by_route["calculate_teu"]["business_result"]["teu"] == 4
assert results_by_route["rule_precheck"]["business_result"]["mock"] is True
assert results_by_route["rule_precheck"]["business_result"]["decision"] == "PASS"
assert results_by_route["clarify"]["business_result"]["type"] == "clarification"
assert SERVICE_CALLS["BK-DEMO-006"] == 1
print("06 LLM Router -> three branches -> response LLM done")


<!-- codex:p0:06 -->
## P0 进阶 · RetryPolicy、异常分类与节点缓存

路由正确不代表外部服务可靠。生产图需要区分：

- `TimeoutError` 等暂态错误：在有限预算内自动重试；
- `ValueError` 等输入错误：立即失败，不应重试；
- 成功且适合复用的只读结果：按输入缓存，并设置 TTL。

主流程中的规则预审节点返回 mock 成功值。为了单独演示 `RetryPolicy` 如何看到真实异常，下面另建一个会抛错的可靠性示例图，不改变主 Router 的三分支路径。

In [ ]:
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy, RetryPolicy


class ReliabilityState(TypedDict, total=False):
    booking_id: str
    result: str


RELIABILITY_CALLS = defaultdict(int)


def protected_rule_lookup(state: ReliabilityState) -> dict:
    booking_id = state["booking_id"]
    RELIABILITY_CALLS[booking_id] += 1

    if booking_id == "INVALID":
        raise ValueError("booking_id 格式无效；输入错误不应自动重试")
    if RELIABILITY_CALLS[booking_id] < 3:
        raise TimeoutError(f"transient timeout: {booking_id}")
    return {"result": f"rule-ok:{booking_id}"}


reliability_builder = StateGraph(ReliabilityState)
reliability_builder.add_node(
    "protected_rule_lookup",
    protected_rule_lookup,
    retry_policy=RetryPolicy(
        # 首次失败后等待 0.01 秒再重试；演示中缩短间隔，避免运行太久
        initial_interval=0.01,
        # 每次重试的等待时间乘以该系数；1.0 表示保持固定间隔，不做指数退避
        backoff_factor=1.0,
        # 单次重试最多等待 0.01 秒，防止退避时间无限增长
        max_interval=0.01,
        # 最多尝试 3 次（包含第一次调用），因此最多发生 2 次重试
        max_attempts=3,
        # 关闭随机抖动，让每次等待时间固定，便于课堂观察和测试
        jitter=False,
        # 只有 TimeoutError 才会触发重试；ValueError 等输入错误会立即抛出
        retry_on=TimeoutError,
    ),
    cache_policy=CachePolicy(ttl=60),
)
reliability_builder.add_edge(START, "protected_rule_lookup")
reliability_builder.add_edge("protected_rule_lookup", END)
reliability_app = reliability_builder.compile(cache=InMemoryCache())

first_lookup = reliability_app.invoke({"booking_id": "BK-RETRY-001"})
calls_after_success = RELIABILITY_CALLS["BK-RETRY-001"]
cached_lookup = reliability_app.invoke({"booking_id": "BK-RETRY-001"})

print("first result =", first_lookup["result"])
print("calls after retry success =", calls_after_success)
print("calls after cached invoke =", RELIABILITY_CALLS["BK-RETRY-001"])

assert calls_after_success == 3
assert cached_lookup["result"] == first_lookup["result"]
assert RELIABILITY_CALLS["BK-RETRY-001"] == 3, "第二次相同输入应命中缓存"

try:
    reliability_app.invoke({"booking_id": "INVALID"})
except ValueError as exc:
    print("non-retryable error =", exc)
else:
    raise AssertionError("ValueError 应直接失败")

assert RELIABILITY_CALLS["INVALID"] == 1
print("06 retry classification + cache ok")

<!-- codex:checklist -->
---

## 练习任务 Checklist

完成后逐项勾选：

- [ ] 让 LLM Router 分别输出 `calculate_teu`、`rule_precheck` 与 `clarify` 结构化意图。
- [ ] 确认 TEU 分支调用 `calculate_teu` 工具，规则预审分支返回 mock 服务结果。
- [ ] 准备一个信息不足的输入，验证流程进入澄清节点。
- [ ] 解释 `path_map` 如何把模型决策映射到实际节点。
- [ ] 确认三条分支都汇合到 `response_llm` 生成最终输出。
- [ ] 对 `TimeoutError` 设置有限次数 RetryPolicy，并确认最终调用次数。
- [ ] 证明 `ValueError` 不会被自动重试。
- [ ] 对相同只读输入执行两次，证明第二次命中缓存。

**交付证据：**三条路由 Trace、retry 计数、non-retryable 异常、缓存计数。